# Minimal warehouse: three facts, three aggregates, one dashboard

This is the **minimal version** of the pizza shop warehouse, agreed with the teacher. It keeps only the sales
star from README section 4, three fact tables, and builds just what the **owner dashboard** needs:

```
POS + Grab ──▶ stg ──▶ dw: 3 facts + 6 dimensions ──▶ agg: 3 tables ──▶ Owner dashboard
```

| Fact table | Grain: one row = | Feeds |
|---|---|---|
| `FCT_ORDER` | one order, any channel | `mart_channel_daily` |
| `FCT_ORDER_LINE` | one menu item on one order | `mart_sales_daily` |
| `FCT_ORDER_LINE_MODIFIER` | one modifier on one order line | `mart_modifier_daily` |

Stock, purchasing, shifts, payments, promotions and delivery events are **out of scope** here; the full model
(`warehouse_design.ipynb`, README) is the roadmap for adding them later on the same conformed dimensions.
The design still follows MADT8102 chapter 5: `_key` surrogate keys, special-case rows instead of NULL keys,
a junk dimension for order status, degenerate order numbers, and drilling across instead of joining facts.

นี่คือ **เวอร์ชันย่อ** ของ warehouse ร้านพิซซ่าที่ตกลงกับอาจารย์ไว้ ใช้แค่ star การขายจาก README section 4 คือ fact สามตัว
และสร้างเฉพาะสิ่งที่ **แดชบอร์ดของเจ้าของร้าน** ต้องใช้ ส่วนสต็อก การสั่งซื้อ กะงาน การชำระเงิน โปรโมชัน และเวลาจัดส่ง **อยู่นอกขอบเขต**
โมเดลเต็ม (`warehouse_design.ipynb`, README) คือแผนสำหรับเพิ่มทีหลังบน conformed dimension ชุดเดิม
การออกแบบยังทำตาม MADT8102 บทที่ 5: surrogate key แบบ `_key`, special-case row แทน key ที่เป็น NULL, junk dimension สำหรับสถานะออร์เดอร์,
เลขออร์เดอร์เป็น degenerate dimension และเทียบ fact ด้วย drill across แทนการ join

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

con = duckdb.connect()                     # in-memory: gone when the kernel stops  |  อยู่ในหน่วยความจำ ปิด kernel แล้วหายไป
for schema in ["src", "stg", "dw", "agg"]: # sources, staging, the star, the aggregate tables
    con.execute(f"CREATE SCHEMA {schema}")  # ต้นทาง, staging, star, aggregate table

def q(sql):
    # run SQL and return a DataFrame so Jupyter shows it as a table  |  รัน SQL แล้วคืนเป็น DataFrame ให้ Jupyter แสดงเป็นตาราง
    return con.execute(sql).df()

def load(table, df):
    # copy a pandas DataFrame into a DuckDB table  |  คัดลอก DataFrame ของ pandas ลงเป็นตารางใน DuckDB
    con.register("_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM _tmp")
    con.unregister("_tmp")

print("ready", duckdb.__version__)

ready 2.0.0


## 1 · Sources

Only the two sales sources the three facts need: the **POS** (QR dine-in and phone) and a **delivery app export**
(Grab here; LINE MAN, ShopeeFood and Robinhood follow the same path). Two business days, Friday 4 and Saturday 5 Sep 2026.
The shop trades 10:00–02:00, so the 01:20 order on the 5th belongs to Friday.

ใช้แค่สองแหล่งข้อมูลการขายที่ fact สามตัวต้องใช้: **POS** (QR และโทรศัพท์) และ **ไฟล์ export ของแอปเดลิเวอรี** (ในที่นี้คือ Grab ส่วนเจ้าอื่นเดินเส้นทางเดียวกัน)
ข้อมูลสองวันทำการ ศุกร์ 4 และเสาร์ 5 ก.ย. 2026 ร้านเปิด 10:00–02:00 ออร์เดอร์ตอนตี 1:20 ของวันที่ 5 จึงเป็นของวันศุกร์

In [2]:
# ---- POS: orders and lines, Bangkok time; modifiers arrive as text  |  POS: ออร์เดอร์และบรรทัด เวลากรุงเทพ modifier มาเป็น text ----
pos_orders = pd.DataFrame([
    ("P-1001", "QR",    "2026-09-04 12:10", "m-081"),
    ("P-1002", "PHONE", "2026-09-04 19:30", "m-092"),
    ("P-1003", "QR",    "2026-09-05 01:20", None),     # after midnight: Friday's business day  |  หลังเที่ยงคืน: นับเป็นวันศุกร์
    ("P-1004", "QR",    "2026-09-05 12:45", None),
    ("P-1005", "PHONE", "2026-09-05 20:05", "m-081"),
], columns=["pos_order_id", "channel", "ordered_at", "member_id"])

pos_order_items = pd.DataFrame([
    ("P-1001", 1, "PEP-L",  1, 329, "EXTRA_CHEESE+40"),
    ("P-1001", 2, "COKE",   2,  35, ""),
    ("P-1002", 1, "HAW-L",  1, 319, ""),
    ("P-1002", 2, "GARLIC", 1,  89, ""),
    ("P-1003", 1, "PEP-L",  1, 329, "EXTRA_CHEESE+40;NO_CHILI+0"),
    ("P-1004", 1, "PEP-L",  2, 349, ""),               # price rose to 349 on Saturday  |  วันเสาร์ขึ้นราคาเป็น 349
    ("P-1004", 2, "COKE",   2,  35, ""),
    ("P-1005", 1, "PEP-M",  1, 249, "EXTRA_CHEESE+40"),
    ("P-1005", 2, "HAW-L",  1, 319, ""),
], columns=["pos_order_id", "line_no", "item_id", "qty", "unit_price", "modifiers"])

# ---- Grab: one flat export, order fields repeated per item row, UTC  |  Grab: ไฟล์แบน ฟิลด์ออร์เดอร์ซ้ำทุกแถว เวลาเป็น UTC ----
grab_export = pd.DataFrame([
    ("GF-201", "2026-09-04T11:30:00Z", "Pepperoni Pizza (L)", 1, 379, 25, 146.4),
    ("GF-201", "2026-09-04T11:30:00Z", "Garlic Bread",        1, 109, 25, 146.4),
    ("GF-202", "2026-09-04T18:10:00Z", "Hawaiian Pizza (L)",  1, 369, 25, 110.7),   # 01:10 Bangkok, still Friday
    ("GF-203", "2026-09-05T06:00:00Z", "Pepperoni Pizza (L)", 2, 399, 25, 252.9),
    ("GF-203", "2026-09-05T06:00:00Z", "Coke",                1,  45, 25, 252.9),
], columns=["orderID", "orderTime", "itemName", "quantity", "itemPrice", "deliveryFee", "merchantCommission"])

for name, df in [("src.pos_orders", pos_orders), ("src.pos_order_items", pos_order_items), ("src.grab_export", grab_export)]:
    load(name, df)
print("loaded", len(pos_orders), "POS orders and", grab_export.orderID.nunique(), "Grab orders")

loaded 5 POS orders and 3 Grab orders


## 2 · Dimensions

Six dimensions and one bridge serve the three facts. Every dimension has special-case rows
**0 = Not applicable, 1 = Unknown, 2 = Invalid**, readable flag values, and its hierarchy:

| Dimension | Hierarchy / notes |
|---|---|
| `DIM_DATE` | day › month › quarter › year, on the business day |
| `DIM_CHANNEL` | channel › channel type (Dine-in / Delivery); own channel or aggregator; commission rate |
| `DIM_CUSTOMER` | members on own channels; row 0 for orders with no identified customer |
| `DIM_MENU_ITEM` | SKU › item › category; **SCD2** on price |
| `DIM_MODIFIER` | extra cheese, no chili…; add / remove / substitute |
| `DIM_ORDER_INFO` | junk dimension: order status, cancellation flag |
| `BRG_CHANNEL_MENU_ITEM` | platform item names → our SKU |

dimension หกตัวกับ bridge หนึ่งตัวให้บริการ fact สามตัว ทุก dimension มี special-case row **0 = ไม่เกี่ยวข้อง, 1 = ไม่ทราบ, 2 = ไม่ถูกต้อง**
เก็บ flag เป็นค่าที่อ่านออก และมี hierarchy ของตัวเอง

In [3]:
SPECIAL = [(0, "Not applicable"), (1, "Unknown"), (2, "Invalid")]

def with_special(rows, columns, member, fill=None):
    # prepend the three special-case rows  |  เพิ่ม special-case row สามแถวไว้ข้างหน้า
    fill = fill or {}
    special = [(k, *[fill.get(c, "n/a") for c in columns[1:-1]], label) for k, label in SPECIAL]
    return pd.DataFrame(special + [(*r, member) for r in rows], columns=columns)

con.execute('''
CREATE TABLE dw.dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INT)                      AS date_key,
       CAST(d AS DATE)                                         AS full_date,
       dayname(d)                                              AS day_name,
       CASE WHEN isodow(d) IN (6, 7) THEN 'Weekend' ELSE 'Weekday' END AS weekend_flag,
       monthname(d)                                            AS month_name,
       'Q' || quarter(d) || ' ' || year(d)                     AS quarter,
       year(d)                                                 AS year,
       'Day'                                                   AS row_type
FROM generate_series(DATE '2026-09-01', DATE '2026-09-30', INTERVAL 1 DAY) AS t(d)
UNION ALL
SELECT k, NULL, 'n/a', 'n/a', 'n/a', 'n/a', NULL, label FROM (VALUES (0, 'Not applicable'), (1, 'Unknown')) AS s(k, label)
''')

load("dw.dim_channel", with_special([
    (101, "QR_DINEIN", "QR dine-in", "Dine-in",  "Own channel",  0.0),
    (102, "PHONE",     "Phone",      "Dine-in",  "Own channel",  0.0),
    (103, "GRAB",      "Grab",       "Delivery", "Aggregator",  30.0),
], ["channel_key", "channel_code", "channel_name", "channel_type", "channel_ownership", "commission_rate_pct", "row_type"],
   "Channel", {"commission_rate_pct": 0.0}))

load("dw.dim_customer", with_special([(101, "m-081", "Member"), (102, "m-092", "Member")],
    ["customer_key", "customer_nk", "membership", "row_type"], "Customer"))

far, start, rise = pd.Timestamp("9999-12-31"), pd.Timestamp("2026-01-01"), pd.Timestamp("2026-09-05")
load("dw.dim_menu_item", with_special([
    (101, "PEP-M",  "Pepperoni M",  "pizza", "Medium", 249, start, far),
    (102, "PEP-L",  "Pepperoni L",  "pizza", "Large",  329, start, rise),   # SCD2: Friday's price  |  ราคาวันศุกร์
    (103, "HAW-L",  "Hawaiian L",   "pizza", "Large",  319, start, far),
    (104, "GARLIC", "Garlic Bread", "side",  "n/a",     89, start, far),
    (105, "COKE",   "Coke",         "drink", "n/a",     35, start, far),
    (106, "PEP-L",  "Pepperoni L",  "pizza", "Large",  349, rise,  far),    # SCD2: from Saturday  |  ตั้งแต่วันเสาร์
], ["menu_item_key", "menu_item_nk", "item_name", "category", "size_name", "base_price_thb", "valid_from", "valid_to", "row_type"],
   "Menu item", {"base_price_thb": None, "valid_from": start, "valid_to": far}))

load("dw.dim_modifier", with_special([(101, "EXTRA_CHEESE", "extra cheese", "add"), (102, "NO_CHILI", "no chili", "remove")],
    ["modifier_key", "modifier_nk", "modifier_name", "modifier_type", "row_type"], "Modifier"))

load("dw.dim_order_info", pd.DataFrame([                        # junk dimension  |  junk dimension
    (101, "completed", "Not cancelled"), (102, "cancelled", "Cancelled"),
], columns=["order_info_key", "order_status", "cancellation_flag"]))

load("dw.brg_channel_menu_item", pd.DataFrame([
    (103, "PEP-M", "Pepperoni Pizza (M)"), (103, "PEP-L", "Pepperoni Pizza (L)"), (103, "HAW-L", "Hawaiian Pizza (L)"),
    (103, "GARLIC", "Garlic Bread"), (103, "COKE", "Coke"),
], columns=["channel_key", "menu_item_nk", "platform_item_name"]))

q("SELECT * FROM dw.dim_menu_item ORDER BY menu_item_key")

,menu_item_key,menu_item_nk,item_name,category,size_name,base_price_thb,valid_from,valid_to,row_type
0,0,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Not applicable
1,1,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Unknown
2,2,n/a,n/a,n/a,n/a,NaN,2026-01-01,9999-12-31,Invalid
3,101,PEP-M,Pepperoni M,pizza,Medium,249.0,2026-01-01,9999-12-31,Menu item
4,102,PEP-L,Pepperoni L,pizza,Large,329.0,2026-01-01,2026-09-05,Menu item
5,103,HAW-L,Hawaiian L,pizza,Large,319.0,2026-01-01,9999-12-31,Menu item
6,104,GARLIC,Garlic Bread,side,n/a,89.0,2026-01-01,9999-12-31,Menu item
7,105,COKE,Coke,drink,n/a,35.0,2026-01-01,9999-12-31,Menu item
8,106,PEP-L,Pepperoni L,pizza,Large,349.0,2026-09-05,9999-12-31,Menu item


## 3 · Staging, then the three facts

Staging turns both sources into one shape (UTC → Bangkok, Grab's repeated order fields taken once, Grab names
mapped to our SKU, the modifier text split into rows). Loading a fact then swaps every business key for a
surrogate key. Each fact carries **its own** date, channel and customer keys; none points at another fact.

staging แปลงทั้งสองแหล่งให้เป็นรูปแบบเดียว (UTC → กรุงเทพ, เอาฟิลด์ออร์เดอร์ที่ Grab ซ้ำมาครั้งเดียว, แปลงชื่อเมนูของ Grab เป็น SKU, แยก text ของ modifier เป็นแถว)
แล้วการโหลด fact ก็คือการเปลี่ยน business key เป็น surrogate key fact ทุกตัวมี key วันที่ ช่องทาง ลูกค้าเป็นของตัวเอง ไม่มีตัวไหนชี้ไปที่ fact ตัวอื่น

In [4]:
con.execute('''
CREATE TABLE stg.order_header AS
SELECT CASE channel WHEN 'QR' THEN 'QR_DINEIN' ELSE channel END AS channel_code,
       pos_order_id AS order_number, CAST(ordered_at AS TIMESTAMP) AS order_ts, member_id AS customer_nk,
       0.0 AS delivery_fee_thb, 0.0 AS commission_amount_thb
FROM src.pos_orders
UNION ALL
SELECT 'GRAB', orderID, MAX(timezone('Asia/Bangkok', CAST(orderTime AS TIMESTAMPTZ))), NULL,
       MAX(deliveryFee), MAX(merchantCommission)                -- repeated per item: take it once  |  ซ้ำทุกแถว: เอาครั้งเดียว
FROM src.grab_export GROUP BY orderID
''')
con.execute('''
CREATE TABLE stg.order_line AS
SELECT h.channel_code, i.pos_order_id AS order_number, i.line_no AS line_number, i.item_id AS menu_item_nk,
       i.qty AS quantity, i.unit_price AS unit_price_thb
FROM src.pos_order_items i JOIN stg.order_header h ON h.order_number = i.pos_order_id
UNION ALL
SELECT 'GRAB', g.orderID, row_number() OVER (PARTITION BY g.orderID ORDER BY g.itemName), m.menu_item_nk, g.quantity, g.itemPrice
FROM src.grab_export g
LEFT JOIN dw.brg_channel_menu_item m ON m.channel_key = 103 AND m.platform_item_name = g.itemName
''')
con.execute('''
CREATE TABLE stg.line_modifier AS                               -- "EXTRA_CHEESE+40;NO_CHILI+0" -> one row each
SELECT h.channel_code, s.order_number, s.line_number, split_part(s.m, '+', 1) AS modifier_nk, 1 AS quantity,
       CAST(split_part(s.m, '+', 2) AS DECIMAL(10, 2)) AS price_delta_thb
FROM (SELECT pos_order_id AS order_number, line_no AS line_number, unnest(string_split(modifiers, ';')) AS m
      FROM src.pos_order_items WHERE modifiers <> '') s
JOIN stg.order_header h USING (order_number)
''')

# the keys every order-level fact reuses  |  key ที่ fact ระดับออร์เดอร์ทุกตัวใช้ร่วมกัน
con.execute('''
CREATE TABLE stg.order_keys AS
SELECT h.*, c.channel_key,
       CAST(strftime(h.order_ts - INTERVAL 10 HOUR, '%Y%m%d') AS INT) AS date_key,      -- business-day rule, once
       COALESCE(cu.customer_key, 0)                                  AS customer_key,  -- 0 = Not applicable, never NULL
       101                                                           AS order_info_key
FROM stg.order_header h
JOIN dw.dim_channel c ON c.channel_code = h.channel_code
LEFT JOIN dw.dim_customer cu ON cu.customer_nk = h.customer_nk
''')

# FCT_ORDER: one row per order  |  หนึ่งแถวต่อหนึ่งออร์เดอร์
con.execute('''
CREATE TABLE dw.fct_order AS
WITH lt AS (SELECT order_number, SUM(quantity * unit_price_thb) AS items_thb FROM stg.order_line GROUP BY 1),
     mt AS (SELECT order_number, SUM(quantity * price_delta_thb) AS mods_thb FROM stg.line_modifier GROUP BY 1)
SELECT k.date_key, k.channel_key, k.customer_key, k.order_info_key, k.order_number, k.order_ts,
       lt.items_thb + COALESCE(mt.mods_thb, 0)                           AS gross_amount_thb,
       k.delivery_fee_thb, k.commission_amount_thb,
       lt.items_thb + COALESCE(mt.mods_thb, 0) - k.commission_amount_thb AS net_revenue_thb
FROM stg.order_keys k JOIN lt USING (order_number) LEFT JOIN mt USING (order_number)
''')

# FCT_ORDER_LINE: one menu item on one order, as-of SCD2 lookup  |  เมนูหนึ่งรายการในออร์เดอร์หนึ่งใบ lookup แบบ as-of
con.execute('''
CREATE TABLE dw.fct_order_line AS
SELECT k.date_key, k.channel_key, k.customer_key, COALESCE(mi.menu_item_key, 1) AS menu_item_key,   -- 1 = Unknown
       k.order_info_key, l.order_number, l.line_number,
       l.quantity, l.unit_price_thb, l.quantity * l.unit_price_thb AS line_net_thb
FROM stg.order_line l
JOIN stg.order_keys k USING (order_number)
JOIN dw.dim_date d ON d.date_key = k.date_key
LEFT JOIN dw.dim_menu_item mi ON mi.menu_item_nk = l.menu_item_nk AND mi.row_type = 'Menu item'
      AND d.full_date >= mi.valid_from AND d.full_date < mi.valid_to
''')

# FCT_ORDER_LINE_MODIFIER: one modifier on one order line  |  modifier หนึ่งตัวบนบรรทัดหนึ่งบรรทัด
con.execute('''
CREATE TABLE dw.fct_order_line_modifier AS
SELECT l.date_key, l.channel_key, l.menu_item_key, COALESCE(md.modifier_key, 1) AS modifier_key,
       s.order_number, s.line_number, s.quantity, s.price_delta_thb
FROM stg.line_modifier s
JOIN dw.fct_order_line l USING (order_number, line_number)
LEFT JOIN dw.dim_modifier md ON md.modifier_nk = s.modifier_nk
''')

for t in ["fct_order", "fct_order_line", "fct_order_line_modifier"]:
    n, unknown = con.execute(f"SELECT count(*), count(*) FILTER (WHERE {'menu_item_key = 1' if t != 'fct_order' else 'false'}) FROM dw.{t}").fetchone()
    print(f"{t:26s} {n:3d} rows   parked on Unknown: {unknown}")
q("SELECT * FROM dw.fct_order ORDER BY order_ts")

fct_order                    8 rows   parked on Unknown: 0
fct_order_line              14 rows   parked on Unknown: 0
fct_order_line_modifier      4 rows   parked on Unknown: 0


,date_key,channel_key,customer_key,order_info_key,order_number,order_ts,gross_amount_thb,delivery_fee_thb,commission_amount_thb,net_revenue_thb
0,20260904,101,101,101,P-1001,2026-09-04 12:10:00,439.0,0.0,0.0,439.0
1,20260904,103,0,101,GF-201,2026-09-04 18:30:00,488.0,25.0,146.4,341.6
2,20260904,102,102,101,P-1002,2026-09-04 19:30:00,408.0,0.0,0.0,408.0
3,20260904,103,0,101,GF-202,2026-09-05 01:10:00,369.0,25.0,110.7,258.3
4,20260904,101,0,101,P-1003,2026-09-05 01:20:00,369.0,0.0,0.0,369.0
5,20260905,101,0,101,P-1004,2026-09-05 12:45:00,768.0,0.0,0.0,768.0
6,20260905,103,0,101,GF-203,2026-09-05 13:00:00,843.0,25.0,252.9,590.1
7,20260905,102,101,101,P-1005,2026-09-05 20:05:00,608.0,0.0,0.0,608.0


## 4 · Three aggregate tables, one per fact

Each aggregate table is built only from the stars, adds no business rule of its own, and is shaped for one part
of the owner dashboard. Ratios such as average order value are *not* stored; the dashboard computes them from the
additive parts (net revenue and order count), so they stay correct at any level.

| Aggregate table | Grain: one row = | Built from | Dashboard tile |
|---|---|---|---|
| `mart_channel_daily` | business date × channel | `FCT_ORDER` | Net revenue by channel, commission paid, average order value |
| `mart_sales_daily` | business date × channel × SKU | `FCT_ORDER_LINE` | Top items, units and sales by item |
| `mart_modifier_daily` | business date × channel × modifier | `FCT_ORDER_LINE_MODIFIER` | Add-on revenue; extra-cheese rate (drilled across with `mart_sales_daily`) |

aggregate table แต่ละตัวสร้างจาก star เท่านั้น ไม่เพิ่มกฎธุรกิจของตัวเอง และจัดรูปให้ตอบแดชบอร์ดของเจ้าของร้านหนึ่งส่วน
อัตราส่วนอย่างยอดเฉลี่ยต่อออร์เดอร์ *ไม่* ถูกเก็บไว้ แดชบอร์ดคำนวณจากส่วนที่บวกได้ (รายได้สุทธิและจำนวนออร์เดอร์) จึงถูกต้องทุกระดับ

In [5]:
con.execute('''
CREATE TABLE agg.mart_channel_daily AS
SELECT d.full_date AS business_date, d.day_name, d.weekend_flag, c.channel_name, c.channel_type, c.channel_ownership,
       count(*)                     AS orders,
       SUM(o.gross_amount_thb)      AS gross_sales_thb,
       SUM(o.commission_amount_thb) AS commission_thb,
       SUM(o.net_revenue_thb)       AS net_revenue_thb
FROM dw.fct_order o
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_order_info oi USING (order_info_key)
WHERE oi.cancellation_flag = 'Not cancelled'
GROUP BY ALL
''')
con.execute('''
CREATE TABLE agg.mart_sales_daily AS
SELECT d.full_date AS business_date, c.channel_name, mi.menu_item_nk, mi.item_name, mi.category, mi.size_name,
       SUM(l.quantity)                  AS units_sold,
       count(DISTINCT l.order_number)   AS orders,
       SUM(l.line_net_thb)              AS item_sales_thb
FROM dw.fct_order_line l
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_menu_item mi USING (menu_item_key) JOIN dw.dim_order_info oi USING (order_info_key)
WHERE oi.cancellation_flag = 'Not cancelled'
GROUP BY ALL
''')
con.execute('''
CREATE TABLE agg.mart_modifier_daily AS
SELECT d.full_date AS business_date, c.channel_name, md.modifier_name, md.modifier_type, mi.category,
       SUM(m.quantity)                       AS times_applied,
       SUM(m.quantity * m.price_delta_thb)   AS modifier_revenue_thb
FROM dw.fct_order_line_modifier m
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_modifier md USING (modifier_key) JOIN dw.dim_menu_item mi USING (menu_item_key)
GROUP BY ALL
''')
for t in ["mart_channel_daily", "mart_sales_daily", "mart_modifier_daily"]:
    print(f"\n{t}")
    display(q(f"SELECT * FROM agg.{t} ORDER BY ALL"))


mart_channel_daily


,business_date,day_name,weekend_flag,channel_name,channel_type,channel_ownership,orders,gross_sales_thb,commission_thb,net_revenue_thb
0,2026-09-04,Friday,Weekday,Grab,Delivery,Aggregator,2,857.0,257.1,599.9
1,2026-09-04,Friday,Weekday,Phone,Dine-in,Own channel,1,408.0,0.0,408.0
2,2026-09-04,Friday,Weekday,QR dine-in,Dine-in,Own channel,2,808.0,0.0,808.0
3,2026-09-05,Saturday,Weekend,Grab,Delivery,Aggregator,1,843.0,252.9,590.1
4,2026-09-05,Saturday,Weekend,Phone,Dine-in,Own channel,1,608.0,0.0,608.0
5,2026-09-05,Saturday,Weekend,QR dine-in,Dine-in,Own channel,1,768.0,0.0,768.0



mart_sales_daily


,business_date,channel_name,menu_item_nk,item_name,category,size_name,units_sold,orders,item_sales_thb
0,2026-09-04,Grab,GARLIC,Garlic Bread,side,n/a,1.0,1,109.0
1,2026-09-04,Grab,HAW-L,Hawaiian L,pizza,Large,1.0,1,369.0
2,2026-09-04,Grab,PEP-L,Pepperoni L,pizza,Large,1.0,1,379.0
3,2026-09-04,Phone,GARLIC,Garlic Bread,side,n/a,1.0,1,89.0
4,2026-09-04,Phone,HAW-L,Hawaiian L,pizza,Large,1.0,1,319.0
5,2026-09-04,QR dine-in,COKE,Coke,drink,n/a,2.0,1,70.0
6,2026-09-04,QR dine-in,PEP-L,Pepperoni L,pizza,Large,2.0,2,658.0
7,2026-09-05,Grab,COKE,Coke,drink,n/a,1.0,1,45.0
8,2026-09-05,Grab,PEP-L,Pepperoni L,pizza,Large,2.0,1,798.0
9,2026-09-05,Phone,HAW-L,Hawaiian L,pizza,Large,1.0,1,319.0



mart_modifier_daily


,business_date,channel_name,modifier_name,modifier_type,category,times_applied,modifier_revenue_thb
0,2026-09-04,QR dine-in,extra cheese,add,pizza,2.0,80.0
1,2026-09-04,QR dine-in,no chili,remove,pizza,1.0,0.0
2,2026-09-05,Phone,extra cheese,add,pizza,1.0,40.0


## 5 · The owner dashboard

The only consumer. Every tile reads the aggregate tables, never the facts, and never the sources.

ผู้ใช้มีแค่กลุ่มเดียว ทุกช่องบนแดชบอร์ดอ่านจาก aggregate table ไม่อ่านจาก fact และไม่อ่านจากต้นทาง

In [6]:
print("1 · Net revenue by channel (what the shop actually keeps)  |  รายได้สุทธิตามช่องทาง")
display(q('''
SELECT business_date, day_name, channel_name, orders, gross_sales_thb, commission_thb, net_revenue_thb,
       round(net_revenue_thb / orders, 2) AS avg_net_per_order_thb      -- ratio from additive parts
FROM agg.mart_channel_daily ORDER BY business_date, net_revenue_thb DESC
'''))

print("2 · Commission paid to platforms, and its share of gross  |  ค่าคอมมิชชันที่จ่ายให้แพลตฟอร์ม")
display(q('''
SELECT business_date, SUM(gross_sales_thb) AS gross_thb, SUM(commission_thb) AS commission_thb,
       round(100 * SUM(commission_thb) / SUM(gross_sales_thb), 1) AS commission_pct_of_gross
FROM agg.mart_channel_daily GROUP BY 1 ORDER BY 1
'''))

print("3 · Top items, both days  |  เมนูขายดี")
display(q('''
SELECT item_name, size_name, SUM(units_sold) AS units, SUM(item_sales_thb) AS item_sales_thb
FROM agg.mart_sales_daily GROUP BY ALL ORDER BY units DESC, item_sales_thb DESC LIMIT 5
'''))

print("4 · Add-ons: extra cheese per pizza sold (drill across two aggregates)  |  สัดส่วนชีสเพิ่มต่อพิซซ่า")
display(q('''
WITH pizzas AS (SELECT business_date, SUM(units_sold) AS pizzas FROM agg.mart_sales_daily
                WHERE category = 'pizza' GROUP BY 1),
     cheese AS (SELECT business_date, SUM(times_applied) AS extra_cheese, SUM(modifier_revenue_thb) AS addon_revenue_thb
                FROM agg.mart_modifier_daily WHERE modifier_name = 'extra cheese' GROUP BY 1)
SELECT p.business_date, p.pizzas, c.extra_cheese, c.addon_revenue_thb,
       round(100.0 * c.extra_cheese / p.pizzas, 1) AS extra_cheese_pct
FROM pizzas p LEFT JOIN cheese c USING (business_date) ORDER BY 1
'''))

1 · Net revenue by channel (what the shop actually keeps)  |  รายได้สุทธิตามช่องทาง


,business_date,day_name,channel_name,orders,gross_sales_thb,commission_thb,net_revenue_thb,avg_net_per_order_thb
0,2026-09-04,Friday,QR dine-in,2,808.0,0.0,808.0,404.00
1,2026-09-04,Friday,Grab,2,857.0,257.1,599.9,299.95
2,2026-09-04,Friday,Phone,1,408.0,0.0,408.0,408.00
3,2026-09-05,Saturday,QR dine-in,1,768.0,0.0,768.0,768.00
4,2026-09-05,Saturday,Phone,1,608.0,0.0,608.0,608.00
5,2026-09-05,Saturday,Grab,1,843.0,252.9,590.1,590.10


2 · Commission paid to platforms, and its share of gross  |  ค่าคอมมิชชันที่จ่ายให้แพลตฟอร์ม


,business_date,gross_thb,commission_thb,commission_pct_of_gross
0,2026-09-04,2073.0,257.1,12.4
1,2026-09-05,2219.0,252.9,11.4


3 · Top items, both days  |  เมนูขายดี


,item_name,size_name,units,item_sales_thb
0,Pepperoni L,Large,7.0,2533.0
1,Coke,n/a,5.0,185.0
2,Hawaiian L,Large,3.0,1007.0
3,Garlic Bread,n/a,2.0,198.0
4,Pepperoni M,Medium,1.0,249.0


4 · Add-ons: extra cheese per pizza sold (drill across two aggregates)  |  สัดส่วนชีสเพิ่มต่อพิซซ่า


,business_date,pizzas,extra_cheese,addon_revenue_thb,extra_cheese_pct
0,2026-09-04,5.0,2.0,80.0,40.0
1,2026-09-05,6.0,1.0,40.0,16.7


Tile 4 is a **drill across**: each aggregate is summarized to the business date first, then the two are combined.
Joining `FCT_ORDER_LINE` to `FCT_ORDER_LINE_MODIFIER` row by row and summing would double-count any line with two
modifiers (P-1003 has extra cheese *and* no chili).

### What's out of scope, and how it comes back

| Later | Adds | Reuses |
|---|---|---|
| Stock and reorder | `FCT_INVENTORY_MOVEMENT`, `FCT_INVENTORY_COUNT`, `BRG_RECIPE` | `DIM_DATE`, `DIM_MENU_ITEM`, `DIM_MODIFIER` |
| Delivery times | `FCT_ORDER_EVENT` | `DIM_DATE`, `DIM_CHANNEL` |
| Labour | `FCT_SHIFT`, `DIM_STAFF` | `DIM_DATE` |
| Payments, promotions | `FCT_PAYMENT`, `FCT_ORDER(_LINE)_PROMOTION` | `DIM_DATE`, `DIM_CHANNEL`, `DIM_CUSTOMER`, `DIM_MENU_ITEM` |

Because the dimensions are conformed, each addition is a new star next to these three, not a redesign.

ช่องที่ 4 คือการ **drill across**: สรุปแต่ละ aggregate ให้อยู่ระดับวันทำการก่อน แล้วค่อยรวมกัน ถ้า join `FCT_ORDER_LINE` กับ `FCT_ORDER_LINE_MODIFIER` ทีละแถวแล้ว sum
บรรทัดที่มี modifier สองตัวจะถูกนับซ้ำ (P-1003 มีทั้งชีสเพิ่มและไม่ใส่พริก)
ส่วนที่อยู่นอกขอบเขต (สต็อก เวลาจัดส่ง ค่าแรง การชำระเงิน โปรโมชัน) เพิ่มกลับมาได้เป็น star ใหม่ข้าง ๆ fact สามตัวนี้ โดยใช้ conformed dimension ชุดเดิม ไม่ต้องออกแบบใหม่

In [7]:
con.close()